# 1. Generate and inspect noisy syndrome data

This notebook checks the Steane syndrome convention, creates the train/validation/test files only if they do not already exist, and inspects a few trajectories. Later notebooks reuse these exact files.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().resolve()))
from main import run_self_tests, save_splits, load_split, calculate_syndrome, correction_labels

print(run_self_tests())
DATA_DIR = Path("data")
expected = [DATA_DIR / "train.npz", DATA_DIR / "validation.npz", DATA_DIR / "test.npz"]
if not all(path.exists() for path in expected):
    save_splits(DATA_DIR, overwrite=False)
else:
    print("Saved dataset files already exist; reusing them without regeneration.")

In [ ]:
train = load_split("data/train.npz")
validation = load_split("data/validation.npz")
test = load_split("data/test.npz")
for split_name, split in [("train", train), ("validation", validation), ("test", test)]:
    print(split_name, {key: value.shape for key, value in split.items()})

## Inspect one trajectory

Each sample contains three observed and three true six-bit syndromes. The labels are reference single-qubit labels derived from the final true syndrome; they are not a unique description of arbitrary multi-qubit physical errors.

In [ ]:
example = 0
print("p, q:", train["p"][example], train["q"][example])
print("True syndromes (3 rounds):\n", train["true_syndromes"][example])
print("Observed syndromes (3 rounds):\n", train["observed"][example])
print("Final X error vector:", train["final_x"][example])
print("Final Z error vector:", train["final_z"][example])
print("Reference labels (X, Z):", train["labels_x"][example], train["labels_z"][example])

In [ ]:
plt.figure(figsize=(7, 3))
values, counts = np.unique(train["p"], return_counts=True)
plt.bar([str(v) for v in values], counts)
plt.title("Training trajectories by data-error probability")
plt.xlabel("p")
plt.ylabel("Number of trajectories")
plt.tight_layout()
plt.show()

## Summary

The data splits are saved as compressed NPZ files. The following notebooks should load these files instead of regenerating them.